# Minimal dlt example: resource + source -> duckdb

Glossary mapping (https://dlthub.com/docs/general-usage/glossary):
- **resource**: a function that yields records (one table)
- **source**: a group of resources
- **pipeline**: moves data from source into a destination (duckdb here)

In [4]:
import dlt


@dlt.resource(table_name="users", write_disposition="replace", primary_key="id")
def users():
    yield {"id": 1, "name": "Alice", "city": "Taipei"}
    yield {"id": 2, "name": "Bob", "city": "Tokyo"}
    yield {"id": 3, "name": "Carol", "city": "Seoul"}


@dlt.resource(table_name="orders", write_disposition="replace", primary_key="order_id")
def orders():
    yield {"order_id": 100, "user_id": 1, "items": [{"sku": "A1"}, {"sku": "B2"}]}
    yield {"order_id": 101, "user_id": 2, "items": [{"sku": "C3"}]}


@dlt.source
def shop_source():
    return [users, orders]


In [9]:
from dlt.destinations import duckdb

pipeline = dlt.pipeline(
    pipeline_name="shop",
    destination=duckdb("results/shop.duckdb"),  # output path; dlt creates result/ if missing
    dataset_name="shop_data",
)

load_info = pipeline.run(shop_source())
print(load_info)

with pipeline.sql_client() as client:
    with client.execute_query("SELECT * FROM users") as cur:
        df = cur.df()
df

Pipeline shop load step completed in 0.04 seconds
1 load package(s) were loaded to destination duckdb and into dataset shop_data
The duckdb destination used duckdb:////Users/markchang/projects/personal/dlt-practice/results/shop.duckdb location to store data
Load package 1780480061.5519319 is LOADED and contains no failed jobs


,id,name,city,_dlt_load_id,_dlt_id
0,1,Alice,Taipei,1780480061.5519319,IILCh8kiFxlUrA
1,2,Bob,Tokyo,1780480061.5519319,zDFcp/idMEtBUw
2,3,Carol,Seoul,1780480061.5519319,GBPHA15ggvabwg
